# 05 — Collect Community Breadth (v2 Panel)

For each user in the v2 DiD panel, queries Arctic Shift for their subreddit activity
during the study window:
```
GET /api/users/interactions/subreddits?author={user}&after=2023-08-01&before=2025-07-31&limit=100
```

**Community breadth** = # distinct subreddits *outside* r/gradadmissions.

**Strategy:** Reuse the existing `data/processed/user_community_breadth.parquet`
(25,305 users from the old pipeline) and only query Arctic Shift for users
in the new v2 panel not already covered.

**Inputs:**
- `data/processed_v2/panel_scores_v2.parquet` (from notebook 04)
- `data/processed/user_community_breadth.parquet` (existing breadth cache)

**Output:** `data/processed_v2/user_community_breadth_v2.parquet`

**Runtime:** Only new users are fetched. Safe to interrupt — progress saved every 500 users.

In [1]:
import json
import time
import numpy as np
import pandas as pd
import requests
from pathlib import Path

ROOT        = Path('..').resolve()
DATA_DIR    = ROOT / 'data' / 'processed'
DATA_V2     = ROOT / 'data' / 'processed_v2'

PANEL_PATH      = DATA_V2 / 'panel_scores_v2.parquet'
OLD_BREADTH     = DATA_DIR / 'user_community_breadth.parquet'
CHECKPOINT_PATH = DATA_V2 / 'breadth_checkpoint_v2.jsonl'
OUT_PATH        = DATA_V2 / 'user_community_breadth_v2.parquet'

BASE_URL     = 'https://arctic-shift.photon-reddit.com'
AFTER_DATE   = '2023-08-01'
BEFORE_DATE  = '2025-07-31'
RATE_LIMIT_SEC   = 0.4
CHECKPOINT_EVERY = 500

print('Panel path:     ', PANEL_PATH)
print('Old breadth:    ', OLD_BREADTH)
print('Checkpoint:     ', CHECKPOINT_PATH)
print('Output:         ', OUT_PATH)

Panel path:      /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/panel_scores_v2.parquet
Old breadth:     /media/ayush/F/Coding/CS598_Research_Project/data/processed/user_community_breadth.parquet
Checkpoint:      /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/breadth_checkpoint_v2.jsonl
Output:          /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/user_community_breadth_v2.parquet


## 1) Identify users to fetch

In [2]:
# Load v2 panel
panel = pd.read_parquet(PANEL_PATH)
v2_users = set(panel['author'])
print(f'v2 panel users:             {len(v2_users):,}')

# Load existing breadth cache
old_breadth = pd.read_parquet(OLD_BREADTH)
old_covered = set(old_breadth[old_breadth['status'] == 'ok']['author'])
print(f'Existing breadth (ok):      {len(old_covered):,}')

# Users we still need to fetch
to_fetch = sorted(v2_users - old_covered)
print(f'New users to fetch:         {len(to_fetch):,}')
print(f'Coverage from cache:        {100 * len(v2_users & old_covered) / len(v2_users):.1f}%')

v2 panel users:             2,014
Existing breadth (ok):      25,305
New users to fetch:         522
Coverage from cache:        74.1%


## 2) Load checkpoint (resume if interrupted)

In [3]:
new_fetched = {}
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH) as f:
        for line in f:
            rec = json.loads(line.strip())
            new_fetched[rec['author']] = rec
    print(f'Resuming — {len(new_fetched):,} new users already in checkpoint')
else:
    print('Starting fresh')

remaining = [u for u in to_fetch if u not in new_fetched]
print(f'Remaining to fetch: {len(remaining):,}')
print(f'Estimated time: ~{len(remaining) * RATE_LIMIT_SEC / 3600:.1f} hours')

Starting fresh
Remaining to fetch: 522
Estimated time: ~0.1 hours


## 3) Fetch community breadth for new users

In [4]:
def fetch_breadth(author: str, session: requests.Session) -> dict:
    """Query Arctic Shift for a user's subreddit interaction counts."""
    url = f'{BASE_URL}/api/users/interactions/subreddits'
    params = {'author': author, 'after': AFTER_DATE, 'before': BEFORE_DATE, 'limit': 100}
    try:
        r = session.get(url, params=params, timeout=15)
        rl_remaining = r.headers.get('X-RateLimit-Remaining')
        rl_reset     = r.headers.get('X-RateLimit-Reset')

        if r.status_code == 200:
            data = r.json().get('data', [])
            exclude = {'gradadmissions', f'u_{author}'}
            subs = [d['subreddit'] for d in data if d.get('subreddit') not in exclude]
            return {'author': author, 'community_breadth': len(subs),
                    'subreddits': subs, 'status': 'ok',
                    'rl_remaining': rl_remaining, 'rl_reset': rl_reset}
        elif r.status_code == 429:
            wait = 10
            if rl_reset:
                try:
                    wait = max(0, float(rl_reset) - time.time()) + 1
                except Exception:
                    pass
            print(f'  Rate limited — waiting {wait:.0f}s')
            time.sleep(wait)
            return {'author': author, 'community_breadth': None, 'subreddits': [], 'status': 'rate_limited'}
        else:
            return {'author': author, 'community_breadth': None, 'subreddits': [], 'status': f'http_{r.status_code}'}

    except requests.exceptions.Timeout:
        return {'author': author, 'community_breadth': None, 'subreddits': [], 'status': 'timeout'}
    except Exception as e:
        return {'author': author, 'community_breadth': None, 'subreddits': [], 'status': f'error:{e}'}

In [5]:
session = requests.Session()
session.headers.update({'User-Agent': 'CS598-research/1.0 (uiuc; gradadmissions study)'})

n_errors = 0
checkpoint_buf = []

print(f'Fetching {len(remaining):,} users...\n(Safe to interrupt — saves every {CHECKPOINT_EVERY} users)')

for i, author in enumerate(remaining):
    result = fetch_breadth(author, session)
    new_fetched[author] = result
    checkpoint_buf.append(result)

    if result['status'] != 'ok':
        n_errors += 1

    if len(checkpoint_buf) >= CHECKPOINT_EVERY:
        with open(CHECKPOINT_PATH, 'a') as f:
            for rec in checkpoint_buf:
                f.write(json.dumps({k: v for k, v in rec.items() if not k.startswith('rl_')}) + '\n')
        checkpoint_buf = []
        pct = (len(new_fetched) + len(old_covered & v2_users)) / len(v2_users) * 100
        print(f'  [{pct:.1f}%] new: {len(new_fetched):,} | errors: {n_errors}')

    # Dynamic rate limiting
    rl_remaining = result.get('rl_remaining')
    try:
        rl_int = int(rl_remaining) if rl_remaining is not None else 999
    except (ValueError, TypeError):
        rl_int = 999
    if rl_int < 5:
        time.sleep(2.0)
    elif rl_int < 20:
        time.sleep(0.8)
    else:
        time.sleep(RATE_LIMIT_SEC)

# Flush remaining buffer
if checkpoint_buf:
    with open(CHECKPOINT_PATH, 'a') as f:
        for rec in checkpoint_buf:
            f.write(json.dumps({k: v for k, v in rec.items() if not k.startswith('rl_')}) + '\n')

print(f'\nFetch complete. New: {len(new_fetched):,}, Errors: {n_errors}')

Fetching 522 users...
(Safe to interrupt — saves every 500 users)
  [98.9%] new: 500 | errors: 3

Fetch complete. New: 522, Errors: 3


## 4) Merge old cache + new fetches → v2 breadth file

In [6]:
# Old cache — keep only v2 panel users
old_keep = old_breadth[old_breadth['author'].isin(v2_users)].copy()
old_keep['subreddits'] = old_keep['subreddits_json'].apply(json.loads)

# New fetches (ok only)
new_ok = pd.DataFrame([r for r in new_fetched.values() if r['status'] == 'ok'])
print(f'From old cache: {len(old_keep):,}  |  From new fetch: {len(new_ok):,}')

if len(new_ok) > 0:
    combined = pd.concat([
        old_keep[['author', 'community_breadth', 'subreddits', 'status']],
        new_ok[['author',   'community_breadth', 'subreddits', 'status']],
    ], ignore_index=True)
else:
    combined = old_keep[['author', 'community_breadth', 'subreddits', 'status']].copy()

# Deduplicate (old cache wins)
combined = combined.drop_duplicates(subset='author', keep='first')

combined['community_breadth_log'] = np.log1p(combined['community_breadth'])
combined['subreddits_json'] = combined['subreddits'].apply(json.dumps)

print(f'Combined unique users: {len(combined):,}')
coverage = 100 * len(set(combined['author']) & v2_users) / len(v2_users)
print(f'Coverage of v2 panel: {coverage:.1f}%')

From old cache: 1,492  |  From new fetch: 519
Combined unique users: 2,011
Coverage of v2 panel: 99.9%


In [7]:
print('Breadth distribution:')
print(combined['community_breadth'].describe().round(2))
print(f"\nUsers with breadth=0: {(combined['community_breadth']==0).sum():,}")

# Exposure breakdown
panel_exp = panel[['author', 'exposed']].drop_duplicates('author')
merged_check = combined.merge(panel_exp, on='author', how='left')
print('\nBreadth by exposure:')
print(merged_check.groupby('exposed')['community_breadth'].describe().round(2))

Breadth distribution:
count    2011.00
mean       38.08
std        33.79
min         0.00
25%        10.00
50%        25.00
75%        61.50
max       100.00
Name: community_breadth, dtype: float64

Users with breadth=0: 15

Breadth by exposure:
          count   mean    std  min    25%   50%    75%    max
exposed                                                      
False    1705.0  35.52  32.88  0.0   9.00  22.0  55.00  100.0
True      306.0  52.37  35.28  2.0  20.25  44.0  97.75   99.0


## 5) Save

In [8]:
out_cols = ['author', 'community_breadth', 'community_breadth_log', 'subreddits_json', 'status']
combined[out_cols].to_parquet(OUT_PATH, index=False)
print(f'Saved {len(combined):,} rows → {OUT_PATH}')

Saved 2,011 rows → /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/user_community_breadth_v2.parquet
